# 🎬 Movie Review Sentiment Classification
**End-to-End NLP & Machine Learning Pipeline**  
*Timeline: Jan 2025 – Feb 2025*  
*Tech Stack: Python, NLP, Machine Learning, Scikit-Learn, TF-IDF*

### Project Overview
- **Objective:** Build a binary sentiment classifier to categorize movie reviews as **positive** or **negative**.
- **Feature Extraction:** Applied **TF-IDF vectorization** with tokenization, contraction expansion, and negation-preserving stopword filtering.
- **Model Benchmarking:** Trained and compared multiple machine learning models (Logistic Regression, LinearSVC, Naive Bayes, Random Forest) to identify the top performer.

## 1. Imports and Setup

In [ ]:
import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add project root to sys.path
PROJECT_ROOT = os.path.dirname(os.getcwd())
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.data_loader import load_dataset
from src.preprocessing import preprocess_text, clean_html, expand_contractions
from src.features import create_vectorizer
from src.train import get_candidate_models, evaluate_model
from src.predict import SentimentPredictor

sns.set_theme(style="whitegrid")
print("Environment initialized successfully.")

## 2. Load Dataset & Exploratory Data Analysis (EDA)

In [ ]:
X_train_raw, X_test_raw, y_train, y_test = load_dataset()
print(f"Training instances: {len(X_train_raw)}")
print(f"Testing instances:  {len(X_test_raw)}")
print("\nClass balance in training set:")
print(y_train.value_counts(normalize=True))

In [ ]:
# Sample reviews
print("Sample Positive Review:")
print(X_train_raw[y_train == 1].iloc[0][:300], "...\n")
print("Sample Negative Review:")
print(X_train_raw[y_train == 0].iloc[0][:300], "...")

## 3. NLP Preprocessing Pipeline
Steps:
1. HTML entity and tag stripping
2. Web URL removal
3. Contraction expansion (e.g. didn't -> did not)
4. Tokenization
5. Stopword filtering with negation retention (`not`, `never`, `no` are preserved)

In [ ]:
sample_raw = "<br />The film didn't disappoint at all! Superb acting: http://imdb.com"
sample_clean = preprocess_text(sample_raw)
print("Original:    ", sample_raw)
print("Preprocessed:", sample_clean)

In [ ]:
# Clean train and test texts
X_train_clean = [preprocess_text(t) for t in X_train_raw]
X_test_clean = [preprocess_text(t) for t in X_test_raw]
print("Preprocessing completed for train and test splits.")

## 4. TF-IDF Feature Extraction (Unigrams + Bigrams)

In [ ]:
vectorizer = create_vectorizer(ngram_range=(1, 2), max_features=10000, min_df=3, sublinear_tf=True)
X_train_tfidf = vectorizer.fit_transform(X_train_clean)
X_test_tfidf = vectorizer.transform(X_test_clean)

print(f"Fitted TF-IDF Feature Matrix: {X_train_tfidf.shape}")
print(f"Vocabulary preview (first 10 terms): {vectorizer.get_feature_names_out()[:10]}")

## 5. Model Training & Benchmarking Comparison

In [ ]:
models = get_candidate_models()
results = []

for name, clf in models.items():
    clf.fit(X_train_tfidf, y_train)
    res = evaluate_model(clf, X_test_tfidf, y_test)
    res["Model"] = name
    results.append(res)
    print(f"{name:25s} | Accuracy: {res['accuracy']:.4f} | F1: {res['f1_score']:.4f} | ROC-AUC: {res['roc_auc']:.4f}")

df_comparison = pd.DataFrame(results)[["Model", "accuracy", "precision", "recall", "f1_score", "roc_auc"]]
df_comparison

## 6. Real-time Inference & Explainability

In [ ]:
predictor = SentimentPredictor()
test_sentence = "A masterclass in storytelling with unforgettable performances and gorgeous cinematography."
prediction = predictor.predict(test_sentence)

print("Review:", test_sentence)
print(f"Predicted Sentiment: {prediction['sentiment']} (Confidence: {prediction['confidence']*100:.1f}%)")
print("Positive Feature Attribution:", prediction["explanation"]["positive_indicators"])